# 15 — PatchTST + DLinear v2 no OD (EF01): protocolo L=2304 + purge/embargo + 5 fatias × 5 seeds

Quinto notebook do protocolo v2 (após 10-baseline-ph, 11-baseline-od, 12-v2-lstnet-ph e 13-v2-lstnet-od). PatchTST nativo + controle DLinear do **05** com as mudanças documentadas abaixo; todo o resto espelha o 05. Janelamento/purge/val **idênticos ao 11/13** (splitter verbatim do §5). 2025 intocado (benchmark futuro).

## Diff exato vs 05 (`notebooks/05-patchtst-od.ipynb`)

| | 05 (v1) | 15 (v2, este notebook) |
|---|---|---|
| Janela | `L=8640 → H=288` (30 d) | **`L=2304 → H=288` (8 d, protocolo v2)** |
| Entrada dos modelos | cauda `LN=2016` da janela, **univariada** | **cauda `LN=2016` da janela, univariada (idêntica; SEM covariáveis — ver limitação)** |
| Arquiteturas | PatchTST (`P=48/S=24`, `d=64`, 3 layers, 4 heads, `ff=128`) + `DLinearLite(k=25)` | **idênticas, verbatim do 05 (1.639.010 + 1.161.794 params)** |
| RevIN | por janela (`γ/β` aprendidos) | **idêntico** |
| Strides treino/val | `8/4` (custo do `L=8640`) | **`4/4, idêntico ao 12/13` (a janela v2 menor permite)** |
| Hiperparâmetros | patch `BATCH=256/LR=1e-3/MAX=60/PAT=10`; dlinear `30/5`, batch 512 | **iguais, por seed × 2 modelos** |
| Val | 4 fatias por data de fim, sem purge | **5 fatias do 11 + purge/embargo ±H (idêntico ao 11/13)** |
| Seeds | 1 (42) | **5 seeds `[42, 7, 123, 2024, 999]` — mesmo treino/early-stopping do 05 por seed** |
| Régua neural | LSTNet-03 recarregada p/ inferência (val 0,1380) | **NÃO recarregada (13 ainda não executado); referência contextual via CSV do 13, com skip se ausente** |
| Réguas v1 (referência) | — | **05: lstnet 0,1380 · patchtst 0,1432 · dlinear 0,1435; régua v2 dos baratos vem do 11 (sazonal-naive val 0,1736)** |

## Por que UNIVARIADO (limitação declarada)

PatchTST/DLinear do 05 são **univariados por construção**: RevIN por janela + atenção/lineares operando só sobre a série de valor — não há canal de covariável na arquitetura (o PatchTST achata `N×d_model` patches do valor; o DLinear decompõe tendência/sazonalidade do valor). As covariáveis solar+Fourier entram no LSTNet-v2 (13) e nas features LGBM-v2; portanto a comparação v1×v2 **embute mudança de protocolo** (L, purge, 5ª fatia), não só o modelo. Declarado como limitação.

## Protocolo v2 (travado, idêntico ao 11/13)

- `L=2304 → H=288` (8 d → 1 d, passo 5 min), interpolação `time` limite 24, descarte de janelas com NaN.
- Val = 5 fatias por data de fim: 19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]** → val 12.960 ([2880, 2880, 2880, 1440, 2880]); treino pós-purge 77.141 + purge 4.320 + NaN-desc 8.109 (do 11 executado; OD só tem micro-outages → 5 fatias cheias).
- Purge/embargo: treino exclui janelas cujo alvo `[fim−H, fim]` intersecte qualquer fatia estendida `±H` (gap mín +289 passos; v1 era −288). Splitter `purge_train`/`signed_gap_steps` **verbatim** do 11/`validate_split.py` — ver §5 (trava por `assert`).

## Execução remota (UM job por vez — 12c/23 GB estouram com concorrência)

- **Solo (máquina livre):** `.venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/15-v2-patchtst-od.ipynb`
- **Compartilhada:** `OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/15-v2-patchtst-od.ipynb`
- Nunca `sleep` dentro do comando remoto (o canal MCP expira); `pkill -f` com o truque `[.]` (ex.: `pkill -f 'nbconvert.*15-v2-patchtst-o[d]'`).
- Tempo estimado: **10 treinos (5 seeds × 2 modelos)** — o 05 levou ~6 min (patchtst, 15 eps, 8.260 janelas/época) + ~10 s (dlinear) solo; a base v2/seed tem ~2,3× janelas/época (stride 8→4 + treino maior) → **~10–18 min/seed de patchtst solo → ~1–1,5 h solo / ~2–3 h compartilhada** no total (dlinear é residual; o early-stopping define as épocas exatas).
- Prophet/ARIMA **não** correm aqui (só PatchTST + DLinear + 3 baratos de referência).

## Saídas (criadas pela execução)

`resultados/15-v2-patchtst-od/`: `metricas_val_por_seed.csv` (10 linhas: 2 modelos × 5 seeds, val pooled) · `metricas_val_media_dp.csv` (média±dp pooled por modelo) · `metricas_por_fatia.csv` (50 linhas: 5 fatias × 2 modelos × 5 seeds) · `metricas_por_fatia_media_dp.csv` (10 linhas: média±dp por fatia×modelo) · `metricas_por_dia.csv` (45 dias-âncora; estende o `metricas_por_dia.csv` do 05 com colunas por seed + média±dp) · `modelos/patchtst_od_s{seed}.pt` + `modelos/dlinear_od_s{seed}.pt` (×5 cada) + `modelos/normalizacao.json` · `figs/` 01-eda/02-limpeza/03-stl/04-forecasts/05-mae/06-val-dias/07-curvas-treino (espelho do 05; 04 com bandas média±dp das 5 seeds, 07 com curvas das 5 seeds + média±dp por modelo). O `README.md` do experimento (formato do 05 + seção “Protocolo v2”) é escrito **após** a execução, com números reais + procedência remota (host + work dir).

Convenção: nada in-place em 00–14 · nada de `src/` · nada de 2025 neste notebook.

In [1]:
import json
import os
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
CSV = ROOT / "dados/treino/ef01-mogi-das-cruzes_oxigenio-dissolvido_2024.csv"
OUT = ROOT / "resultados" / "15-v2-patchtst-od"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# Protocolo v2 (travado, idêntico ao 11/13)
L, H = 2304, 288  # 8 d -> 1 d (passo 5 min)
SEASON = 288
INTERP_LIMIT = 24  # 2 h
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24"),
              ("2024-12-13", "2024-12-22")]
# Arquiteturas do 05 (verbatim; univariadas — ver limitação no cabeçalho)
LN, HN = 2016, 288
PATCH_P, PATCH_S = 48, 24
D_MODEL, NLAYERS, NHEAD, FF = 64, 3, 4, 128
DL_K = 25
BATCH, LR = 256, 1e-3
MAX_EPOCHS, PATIENCE = 60, 10  # patchtst, espelho do 05, por seed
DL_MAX, DL_PAT, DL_BATCH = 30, 5, 512  # dlinear, espelho do 05, por seed
TRAIN_STRIDE, VAL_STRIDE = 4, 4  # idêntico ao 12/13 (o 05 usava 8/4 pelo custo do L=8640)
DROPOUT = 0.1
SEEDS = [42, 7, 123, 2024, 999]  # 42 = padrão do repo; 5 reps p/ média±dp
DEVICE = torch.device("cpu")

print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)
print(f"L={L} H={H} LN={LN} HN={HN} patch(P={PATCH_P}/S={PATCH_S},d={D_MODEL},L={NLAYERS},h={NHEAD}) dlin(k={DL_K}) fatias={len(VAL_SLICES)} seeds={SEEDS}")
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
      "| cpu:", os.cpu_count())
print("OUT:", OUT)

ROOT: /home/marcos/temporal-model | CSV existe: True | torch: 2.14.0+cpu
L=2304 H=288 LN=2016 HN=288 patch(P=48/S=24,d=64,L=3,h=4) dlin(k=25) fatias=5 seeds=[42, 7, 123, 2024, 999]
threads: {'OMP_NUM_THREADS': '<unset>', 'MKL_NUM_THREADS': '<unset>', 'OPENBLAS_NUM_THREADS': '<unset>'} | cpu: 12
OUT: /home/marcos/temporal-model/resultados/15-v2-patchtst-od


## 1. Carga

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "Oxigênio Dissolvido (mg/L)": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 594 (0.6%)


,ds,y
count,105121,104527.000000
mean,2024-07-01 12:00:00,4.491787
min,2024-01-01 00:00:00,0.790000
25%,2024-04-01 06:00:00,2.810000
50%,2024-07-01 12:00:00,4.870000
75%,2024-09-30 18:00:00,6.000000
max,2024-12-31 00:00:00,7.710000
std,NaN,1.725755


## 2. EDA

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("od EF01 2024 — série completa (treino)")
ax[0].set_ylabel("od")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")

maior gap: 334 passos = 27.8 h | gaps > 24 passos: 3


fig salva


## 3. Limpeza

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots = {len(g)*5/60:.1f} h)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 336
  outage 2024-02-02 13:45:00 → 2024-02-02 14:45:00 (13 slots = 1.1 h)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots = 1.1 h)
  outage 2024-03-25 15:30:00 → 2024-03-26 17:15:00 (310 slots = 25.8 h)
fig salva


## 4. ADF + STL (trecho limpo jul–ago)

In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-6.61 p-valor=6.3e-09 → estacionária


fig salva


## 5. Janelamento (L=2304) + val em 5 fatias + purge/embargo ±H

Idêntico ao 11/13: janelas por data de **fim**, descarte com NaN pós-interp, treino = janelas válidas fora da val que **sobrevivem ao purge** (alvo `[fim−H, fim]` sem interseção com qualquer fatia estendida `±H`). Funções `purge_train`/`signed_gap_steps` **verbatim** de `/tmp/v2split/validate_split.py` (ver 11 §5). Trava se o purge falhar (gap < H+1 ou overlap > 0). Esperado OD: treino 77.141 + val 12.960 ([2880, 2880, 2880, 1440, 2880]) + purge 4.320 + NaN-desc 8.109 (do 11 executado; OD só tem micro-outages → 5 fatias cheias).

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
n_desc_nan = int((~ok).sum())
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date

# --- val por data de fim (5 fatias, incl. dez) ---
is_val = np.zeros(len(ends), dtype=bool)
per_slice = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    per_slice.append(int(m.sum()))
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")


def purge_train(ends, is_val, slices):
    """Descarta treino cujo alvo [fim-H, fim] intersecte fatia estendida ±H.
    Verbatim de /tmp/v2split/validate_split.py (H global)."""
    keep = is_val.copy()
    drop = np.zeros(len(ends), dtype=bool)
    for a, b in slices:
        A = pd.Timestamp(a) - pd.Timedelta(minutes=5 * H)          # ini-H
        B = pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5) \
            + pd.Timedelta(minutes=5 * H)                           # fim+H
        tgt0 = ends - pd.Timedelta(minutes=5 * H)
        hit = (~is_val) & (ends >= A) & (tgt0 <= B)  # alvo ∩ [A,B] ≠ ∅
        drop |= hit
    keep[~is_val & ~drop] = True  # treino sobrevivente
    return keep, drop  # keep=True → val ou treino válido


def signed_gap_steps(ends_tr, slices):
    """Distância (passos 5min) do alvo [fim-H,fim] à fatia mais próxima; <0 = overlap.
    Verbatim de /tmp/v2split/validate_split.py."""
    if not len(ends_tr):
        return None
    e = ends_tr.values.astype("datetime64[m]").astype(np.int64)  # min
    best = np.full(len(e), 10 ** 12)
    for a, b in slices:
        A = (pd.Timestamp(a).to_datetime64().astype("datetime64[m]").astype(int))
        B = ((pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5))
             .to_datetime64().astype("datetime64[m]").astype(int))
        s0 = e - H * 5
        ov = (e >= A) & (s0 <= B)
        gap = np.where(e < A, (A - e) // 5, np.where(s0 > B, (s0 - B) // 5, -(np.minimum(e, B) - np.maximum(s0, A)) // 5 - 1))
        best = np.minimum(best, gap)
    return best


keep, drop = purge_train(ends, is_val, VAL_SLICES)
va = np.where(is_val)[0]
tr = np.where(keep & ~is_val)[0]
print(f"treino (pós-purge): {len(tr)} | val: {len(va)} | "
      f"descartadas (NaN): {n_desc_nan} | purge: {int(drop.sum())}")

# --- asserts de cobertura por fatia (OD: só micro-outages → 5 fatias cheias, idem 11) ---
esperado = [288 * ((pd.Timestamp(b) - pd.Timestamp(a)).days + 1) for a, b in VAL_SLICES]
assert per_slice == esperado, f"cobertura por fatia fora do esperado: {per_slice} vs {esperado}"
assert len(va) == sum(esperado) == 12960, f"val total inesperada: {len(va)}"
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
assert int(drop.sum()) > 0, "purge removeu zero janelas — lógica inativa?"

# --- trava do purge: gap mín ≥ H+1 e overlap zero ---
gaps = signed_gap_steps(ends[tr], VAL_SLICES)
print(f"gap mín alvo-treino→val: +{int(gaps.min())} passos (exigido ≥ {H + 1}); "
      f"treino c/ alvo∩val: {int((gaps < 0).sum())}")
assert int((gaps < 0).sum()) == 0, "LEAKAGE: há alvo de treino dentro da val!"
assert int(gaps.min()) >= H + 1, f"purge/embargo falhou: gap {int(gaps.min())} < {H + 1}"

# --- dias-âncora (23:55) na val: 45 = 10+10+10+5+10 ---
daily_mask = (ends.time == pd.Timestamp("23:55").time()) & is_val
daily_idx = np.where(daily_mask)[0]
por_dia_ct = [int(((ends[daily_idx].date >= pd.Timestamp(a).date()) &
                   (ends[daily_idx].date <= pd.Timestamp(b).date())).sum())
              for a, b in VAL_SLICES]
print("dias-âncora na val:", len(daily_idx), "por fatia:", por_dia_ct)
assert len(daily_idx) == 45, f"dias-âncora inesperados: {len(daily_idx)}"
assert por_dia_ct == [10, 10, 10, 5, 10], f"âncoras por fatia: {por_dia_ct}"

fatia 2024-04-19 → 2024-04-28: 2880 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
fatia 2024-12-13 → 2024-12-22: 2880 janelas válidas
treino (pós-purge): 77141 | val: 12960 | descartadas (NaN): 8109 | purge: 4320
gap mín alvo-treino→val: +289 passos (exigido ≥ 289); treino c/ alvo∩val: 0
dias-âncora na val: 45 por fatia: [10, 10, 10, 5, 10]


## 6. Métricas + baselines de referência

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr, Xva, Yva = X[tr], Y[tr], X[va], Y[va]
print("treino rolante (régua v2 dos baratos; v1/05 p/ referência: lstnet 0,1380 · patchtst 0,1432 · dlinear 0,1435):")
print(pd.DataFrame({m: metricas(Ytr, p) for m, p in cheap_preds(Xtr).items()}).T.round(4).to_string())
print("val rolante (5 fatias):")
print(pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T.round(4).to_string())

treino rolante (régua v2 dos baratos; v1/05 p/ referência: lstnet 0,1380 · patchtst 0,1432 · dlinear 0,1435):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.3812  0.5694  8.6819  8.6028
sazonal_naive_288  0.2356  0.3650  6.1279  6.0964
media_movel_288    0.3539  0.4763  8.3817  8.3034
val rolante (5 fatias):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.4280  0.6222  7.7187  7.6334
sazonal_naive_288  0.1736  0.2548  3.3647  3.3567
media_movel_288    0.3631  0.4801  6.6894  6.6394


## 7. Janelas nativas univariadas (cauda LN=2016, SEM covariáveis)

Construção `Wln`/`rowln`/`monta` **idêntica ao 05**: o modelo consome a cauda `LN=2016` (7 d) da janela `L=2304`, só o canal de valor (RevIN por janela dentro do forward). Sem `Tln`/covariáveis — limitação declarada no cabeçalho (covariáveis só no LSTNet-v2/LGBM-v2). `normalizacao.json` registra o modo univariado + arquiteturas + seeds.

In [8]:
val5 = s.to_numpy().astype(np.float32)
Wln = sliding_window_view(val5, LN)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - LN + 1
print(f"Wln {Wln.shape} (esperado (_, {LN}))")
print(f"janelas nativas válidas: {int((rowln >= 0).sum())}/{len(ends)}")
assert Wln.shape[1] == LN, Wln.shape
assert int((rowln >= 0).sum()) == len(ends), "cauda LN fora da grade!"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
json.dump({"mode": "revin-per-window", "LN": LN, "HN": HN, "L": L, "H": H,
           "channels": ["valor"],
           "arch": {"patchtst": {"patch": [PATCH_P, PATCH_S], "d_model": D_MODEL,
                                 "layers": NLAYERS, "heads": NHEAD, "ff": FF},
                    "dlinear": {"k": DL_K}},
           "val_slices": VAL_SLICES, "seeds": SEEDS},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("normalizacao.json salva (univariada: só canal de valor)")


def monta(idxs):
    ii = np.asarray(idxs); r = rowln[ii]
    return Wln[r], Y[ii].astype(np.float32)


Xtr_, Ytr_ = monta(tr[::TRAIN_STRIDE])
Xva_, Yva_ = monta(va[::VAL_STRIDE])
print(f"treino: {Xtr_.shape} (stride {TRAIN_STRIDE}) | val: {Xva_.shape} (stride {VAL_STRIDE})")

Wln (103106, 2016) (esperado (_, 2016))
janelas nativas válidas: 94421/94421
normalizacao.json salva (univariada: só canal de valor)
treino: (19286, 2016) (stride 4) | val: (3240, 2016) (stride 4)


## 8. PatchTST + DLinear — mesmo treino/early-stopping do 05, repetido nas 5 seeds × 2 modelos

Classes **verbatim do 05** (`PatchTST`: patches `P=48/S=24` → proj `d=64` + pos + `TransformerEncoder(3×, 4 heads, ff=128)` + head `(N·d)→288`; `DLinearLite(k=25)`: pool móvel + 2 lineares; ambos com RevIN por janela `γ/β`). Hiperparâmetros espelhados (Adam `LR=1e-3`/MSE; patch `BATCH=256/MAX=60/PAT=10`; dlinear `batch=512/30/5`); por (modelo, seed): fixa `random`/`numpy`/`torch`, DataLoader com `shuffle=True`, salva o melhor `val` em `modelos/{patchtst,dlinear}_od_s{seed}.pt`. Réguas v1 de referência: **05 patchtst 0,1432 · dlinear 0,1435 · lstnet(03) 0,1380**.

In [9]:
class PatchTST(nn.Module):
    def __init__(self):
        super().__init__()
        self.N = (LN - PATCH_P) // PATCH_S + 1
        self.proj = nn.Linear(PATCH_P, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, self.N, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, NHEAD, FF, DROPOUT, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, NLAYERS)
        self.drop = nn.Dropout(DROPOUT)
        self.head = nn.Linear(self.N * D_MODEL, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        z = self.proj(xn.unfold(1, PATCH_P, PATCH_S)) + self.pos
        z = self.enc(self.drop(z))
        y = self.head(self.drop(z.flatten(1)))
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu


class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, HN)
        self.lin_s = nn.Linear(LN, HN)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg + mu


MODELOS = {
    "patchtst": (PatchTST, "patchtst_od_s{seed}.pt", MAX_EPOCHS, PATIENCE, BATCH),
    "dlinear": (DLinearLite, "dlinear_od_s{seed}.pt", DL_MAX, DL_PAT, DL_BATCH),
}
print(f"patchtst params={sum(p.numel() for p in PatchTST().parameters())} (05: 1639010) | "
      f"dlinear params={sum(p.numel() for p in DLinearLite().parameters())} (05: 1161794)")

hists, bests, epochs_best, tempos = {}, {}, {}, {}
t_all = time.time()
for nome, (cls, tpl, max_ep, pat, batch) in MODELOS.items():
    for sd in SEEDS:
        random.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
        model = cls().to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.MSELoss()
        Xtr_v_, Ytr__ = monta(tr[::TRAIN_STRIDE])
        Xva_v_, Yva__ = monta(va[::VAL_STRIDE])
        tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr_v_), torch.from_numpy(Ytr__)),
                               batch_size=batch, shuffle=True)
        va_loader = DataLoader(TensorDataset(torch.from_numpy(Xva_v_), torch.from_numpy(Yva__)),
                               batch_size=512)
        best, patience, hist = float("inf"), 0, {"train": [], "val": []}
        t0 = time.time()
        for ep in range(1, max_ep + 1):
            model.train()
            tl = 0.0
            for xb, yb in tr_loader:
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()
                tl += float(loss.detach()) * len(xb)
            tl /= len(tr_loader.dataset)
            model.eval()
            vl = 0.0
            with torch.no_grad():
                for xb, yb in va_loader:
                    vl += float(loss_fn(model(xb), yb)) * len(xb)
            vl /= len(va_loader.dataset)
            hist["train"].append(tl); hist["val"].append(vl)
            tag = ""
            if vl < best:
                best, patience, best_ep = vl, 0, ep
                torch.save({"state": model.state_dict(), "seed": sd,
                            "cfg": {"ln": LN, "hn": HN, "patch": [PATCH_P, PATCH_S],
                                    "d_model": D_MODEL, "layers": NLAYERS,
                                    "heads": NHEAD, "ff": FF, "dl_k": DL_K}},
                           OUT / "modelos" / tpl.format(seed=sd))
                tag = " *"
            else:
                patience += 1
            print(f"[{nome} s{sd}] ep {ep:02d} train={tl:.4f} val={vl:.4f}{tag}", flush=True)
            if patience >= pat:
                print(f"[{nome} s{sd}] early stopping na ep {ep} (best val={best:.4f} ep {best_ep})")
                break
        dt = time.time() - t0
        hists[(nome, sd)], bests[(nome, sd)] = hist, best
        epochs_best[(nome, sd)], tempos[(nome, sd)] = best_ep, dt
        print(f"[{nome} s{sd}] treino em {dt:.0f}s | melhor val={best:.4f} (ep {best_ep})")
print(f"10 treinos em {time.time()-t_all:.0f}s")
print(pd.DataFrame({"best_val": bests, "best_ep": epochs_best,
                    "train_s": {k: round(v) for k, v in tempos.items()}}).T.round(4).to_string())

patchtst params=1639010 (05: 1639010) | dlinear params=1161794 (05: 1161794)


[patchtst s42] ep 01 train=0.1422 val=0.0688 *


[patchtst s42] ep 02 train=0.0935 val=0.0460 *


[patchtst s42] ep 03 train=0.0844 val=0.0491


[patchtst s42] ep 04 train=0.0754 val=0.0481


...

[dlinear s999] ep 09 train=0.0887 val=0.0462


[dlinear s999] ep 10 train=0.0876 val=0.0445


[dlinear s999] ep 11 train=0.0872 val=0.0441


[dlinear s999] ep 12 train=0.0872 val=0.0447


[dlinear s999] early stopping na ep 12 (best val=0.0440 ep 7)
[dlinear s999] treino em 14s | melhor val=0.0440 (ep 7)
10 treinos em 3720s
         patchtst                                         dlinear                                  
             42        7        123       2024      999      42       7        123     2024    999 
best_val    0.046    0.0444    0.046    0.0466    0.0435   0.0451   0.0442   0.0422   0.043   0.044
best_ep     2.000    2.0000    4.000    5.0000    4.0000   5.0000   6.0000  19.0000  16.000   7.000
train_s   645.000  651.0000  765.000  809.0000  756.0000  12.0000  13.0000  29.0000  25.000  14.000


## 9. Inferência + tabelas (por seed, média±dp pooled e por fatia)

Inferência cheia na val (12.960 origens, sem stride) por (modelo, seed); `metricas_val_por_seed.csv` = pooled por seed (10 linhas), `metricas_val_media_dp.csv` = média±dp pooled por modelo, `metricas_por_fatia.csv` = 5 fatias × 2 modelos × 5 seeds (50 linhas), `metricas_por_fatia_media_dp.csv` = média±dp por fatia×modelo, `metricas_por_dia.csv` = 45 dias-âncora (estende o do 05 com colunas por seed + média±dp). Réguas v1 de referência nos textos: **05 patchtst 0,1432 · dlinear 0,1435 · lstnet(03) 0,1380**; média do 13 lida do CSV quando existir (skip elegante se o 13 ainda não foi executado).

In [10]:
@torch.no_grad()
def prevê(model_, idxs, batch=256):
    model_.eval()
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln[rowln[ii[b:b+batch]]])
        outs.append(model_(xb).numpy())
    return np.concatenate(outs)


MODELOS_N = list(MODELOS)
modelos = {}
for nome, (cls, tpl, *_rest) in MODELOS.items():
    for sd in SEEDS:
        m = cls().to(DEVICE)
        ckpt = torch.load(OUT / "modelos" / tpl.format(seed=sd), map_location="cpu", weights_only=False)
        m.load_state_dict(ckpt["state"]); m.eval()
        modelos[(nome, sd)] = m
print("checkpoints carregados:", sorted(f"{m}/s{s}" for m, s in modelos))

t0 = time.time()
P_va = {(m, sd): prevê(modelos[(m, sd)], va) for m, sd in modelos}  # (12960, 288) cada
P_d = {(m, sd): prevê(modelos[(m, sd)], daily_idx) for m, sd in modelos}  # (45, 288) cada
print(f"inferência val+dias em {time.time()-t0:.0f}s | P_va[patchtst,s42] {P_va[('patchtst', 42)].shape}")

# --- pooled por seed: 2 modelos × 5 seeds ---
rows_seed = []
for nome in MODELOS_N:
    for sd in SEEDS:
        mm = metricas(Yva, P_va[(nome, sd)])
        rows_seed.append({"modelo": nome, "seed": sd, "MAE": mm["MAE"], "RMSE": mm["RMSE"],
                          "MAPE": mm["MAPE"], "sMAPE": mm["sMAPE"],
                          "best_epoch": epochs_best[(nome, sd)], "train_s": round(tempos[(nome, sd)]),
                          "best_val_mse": bests[(nome, sd)]})
tab_seed = pd.DataFrame(rows_seed,
                        columns=["modelo", "seed", "MAE", "RMSE", "MAPE", "sMAPE",
                                 "best_epoch", "train_s", "best_val_mse"]).round(4)
tab_seed.to_csv(OUT / "metricas_val_por_seed.csv", index=False)
assert len(tab_seed) == 10, tab_seed.shape
print("=== val pooled por seed (12.960 origens) ===")
print(tab_seed.to_string(index=False))

# --- média±dp pooled por modelo ---
rows_md = []
for nome in MODELOS_N:
    sub = tab_seed.loc[tab_seed["modelo"] == nome, ["MAE", "RMSE", "MAPE", "sMAPE"]]
    row = {"modelo": nome}
    for c in ["MAE", "RMSE", "MAPE", "sMAPE"]:
        row[c + "_media"] = round(float(sub[c].mean()), 4)
        row[c + "_dp"] = round(float(sub[c].std(ddof=1)), 4)
    rows_md.append(row)
tab_md = pd.DataFrame(rows_md)
tab_md.to_csv(OUT / "metricas_val_media_dp.csv", index=False)
print("=== val pooled média±dp (5 seeds, por modelo) ===")
print(tab_md.to_string(index=False))
for _m in MODELOS_N:
    _r = tab_md.loc[tab_md["modelo"] == _m].iloc[0]
    print(f"v2/{_m}: {float(_r['MAE_media']):.4f}±{float(_r['MAE_dp']):.4f}")
print("Réguas v1 (05, L=8640 4 fatias): patchtst 0,1432 · dlinear 0,1435 · lstnet(03) 0,1380")

# --- contexto 13 (LSTNet-v2 OD): leitura com skip elegante se ausente ---
R13 = ROOT / "resultados" / "13-v2-lstnet-od" / "metricas_val_media_dp.csv"
try:
    print("=== contexto 13 (LSTNet-v2 OD, média±dp) ===")
    print(pd.read_csv(R13).to_string(index=False))
except FileNotFoundError:
    print("contexto 13 ausente (13-v2-lstnet-od ainda não executado) — skip.")

# --- por fatia: 5 × 2 × 5 ---
va_ends = ends[va]
rows_f = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m_va = (va_ends.date >= d0) & (va_ends.date <= d1)
    ii_loc = np.where(m_va)[0]  # posições dentro de va
    for nome in MODELOS_N:
        for sd in SEEDS:
            mm = metricas(Yva[ii_loc], P_va[(nome, sd)][ii_loc])
            rows_f.append({"fatia": f"{a}→{b}", "modelo": nome, "seed": sd, **mm})
tab_f = pd.DataFrame(rows_f, columns=["fatia", "modelo", "seed", "MAE", "RMSE", "MAPE", "sMAPE"]).round(4)
tab_f.to_csv(OUT / "metricas_por_fatia.csv", index=False)
assert len(tab_f) == 50 and tab_f["fatia"].nunique() == 5, tab_f.shape
assert (tab_f["fatia"] == "2024-12-13→2024-12-22").any(), "fatia dez ausente!"
# média±dp por (fatia, modelo): groupby SEM pré-seleção + seleção de coluna ÚNICA por vez
# (nunca groupby(..)[cols] seguido de g[c] — IndexError no pandas ≥2)
g = tab_f.groupby(["fatia", "modelo"])
tab_fmd = pd.DataFrame({c: g[c].mean().round(4).astype(str) + "±" + g[c].std(ddof=1).round(4).astype(str)
                        for c in ["MAE", "RMSE", "MAPE", "sMAPE"]})
tab_fmd.to_csv(OUT / "metricas_por_fatia_media_dp.csv")
print("=== val por fatia — MAE por (modelo, seed) ===")
print(tab_f.pivot_table(index="fatia", columns=["modelo", "seed"], values="MAE").round(4).to_string())
print("=== val por fatia — média±dp ===")
print(tab_fmd.to_string())

# --- por dia-âncora (45 dias; média±dp das 5 seeds por modelo + baratos) ---
Yd = Y[daily_idx]
cp_d = cheap_preds(X[daily_idx])
mae_dia = {nome: np.stack([[mae(Yd[k:k+1], P_d[(nome, sd)][k:k+1])
                            for k in range(len(Yd))] for sd in SEEDS]) for nome in MODELOS_N}  # (5,45) p/ modelo
datas = [str(ends[i].date()) for i in daily_idx]
fatias_d = []
for dt_ in ends[daily_idx].date:
    for a, b in VAL_SLICES:
        if pd.Timestamp(a).date() <= dt_ <= pd.Timestamp(b).date():
            fatias_d.append(f"{a}→{b}"); break
por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cp_d[m][k:k+1]) for k in range(len(Yd))] for m in cp_d},
    index=datas)
for nome in MODELOS_N:
    for j, sd in enumerate(SEEDS):
        por_dia[f"{nome}_s{sd}"] = mae_dia[nome][j]
    por_dia[f"{nome}_media"] = mae_dia[nome].mean(axis=0)
    por_dia[f"{nome}_dp"] = mae_dia[nome].std(axis=0, ddof=1)
por_dia.insert(0, "fatia", fatias_d)
por_dia.to_csv(OUT / "metricas_por_dia.csv")
assert len(por_dia) == 45 and (por_dia["fatia"] == "2024-12-13→2024-12-22").sum() == 10
print("=== val dias-âncora (45): média das 5 seeds vs baratos ===")
print(por_dia[["fatia"] + list(cp_d)
              + [f"{m}_media" for m in MODELOS_N] + [f"{m}_dp" for m in MODELOS_N]].round(4).to_string())
ib = tab_seed["MAE"].idxmin()
print(f"\nMelhor na val pooled: {tab_seed.loc[ib, 'modelo']}_s{int(tab_seed.loc[ib, 'seed'])} = {tab_seed.loc[ib, 'MAE']:.4f}")

checkpoints carregados: ['dlinear/s123', 'dlinear/s2024', 'dlinear/s42', 'dlinear/s7', 'dlinear/s999', 'patchtst/s123', 'patchtst/s2024', 'patchtst/s42', 'patchtst/s7', 'patchtst/s999']


inferência val+dias em 22s | P_va[patchtst,s42] (12960, 288)


=== val pooled por seed (12.960 origens) ===
  modelo  seed    MAE   RMSE   MAPE  sMAPE  best_epoch  train_s  best_val_mse
patchtst    42 0.1564 0.2144 2.9986 2.9864           2      645        0.0460
patchtst     7 0.1522 0.2107 2.9162 2.9172           2      651        0.0444
patchtst   123 0.1577 0.2143 3.0155 3.0170           4      765        0.0460
patchtst  2024 0.1585 0.2159 2.9915 2.9961           5      809        0.0466
patchtst   999 0.1531 0.2084 2.9096 2.9184           4      756        0.0435
 dlinear    42 0.1520 0.2124 2.9024 2.9066           5       12        0.0451
 dlinear     7 0.1492 0.2101 2.8442 2.8424           6       13        0.0442
 dlinear   123 0.1445 0.2052 2.7566 2.7565          19       29        0.0422
 dlinear  2024 0.1474 0.2071 2.8211 2.8157          16       25        0.0430
 dlinear   999 0.1481 0.2097 2.8419 2.8382           7       14        0.0440
=== val pooled média±dp (5 seeds, por modelo) ===
  modelo  MAE_media  MAE_dp  RMSE_media  RMSE_d

=== val por fatia — MAE por (modelo, seed) ===
modelo                dlinear                                 patchtst                                
seed                     7       42      123     999     2024     7       42      123     999     2024
fatia                                                                                                 
2024-04-19→2024-04-28  0.0991  0.1041  0.0958  0.1029  0.1008   0.1020  0.1046  0.1057  0.0977  0.1006
2024-07-20→2024-07-29  0.1140  0.1242  0.1136  0.1123  0.1135   0.1088  0.1132  0.1334  0.1138  0.1510
2024-09-15→2024-09-24  0.1843  0.1806  0.1796  0.1841  0.1813   0.1858  0.1887  0.1810  0.1734  0.1735
2024-11-20→2024-11-24  0.1577  0.1590  0.1544  0.1550  0.1594   0.1685  0.1595  0.1567  0.1920  0.1810
2024-12-13→2024-12-22  0.1952  0.1956  0.1840  0.1898  0.1878   0.2041  0.2174  0.2114  0.2082  0.1976
=== val por fatia — média±dp ===
                                          MAE           RMSE           MAPE          sMAPE
fatia

## 10. Figuras (espelho do 05; 04 com bandas média±dp, 07 com 5 seeds + média±dp por modelo)

In [11]:
# --- 04-forecasts: 3 origens do treino (real × sazonal × patchtst/dlinear média±dp 5 seeds) ---
ks = [0, len(Xtr) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
cp_tr = cheap_preds(Xtr)
for ax, k in zip(axes, ks):
    tf = pd.date_range(ends[tr[k]] - pd.Timedelta(minutes=5*(H-1)), ends[tr[k]], freq="5min")
    ax.plot(tf, Ytr[k], "k-", lw=1.5, label="real")
    ax.plot(tf, cp_tr["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    for nome, ls, al in [("patchtst", "-", 0.9), ("dlinear", ":", 0.7)]:
        Pk = np.stack([prevê(modelos[(nome, sd)], np.array([tr[k]]))[0] for sd in SEEDS])
        mu, dp = Pk.mean(axis=0), Pk.std(axis=0, ddof=1)
        ax.plot(tf, mu, ls, lw=1, alpha=al, label=f"{nome} média 5 seeds")
        ax.fill_between(tf, mu - dp, mu + dp, alpha=0.15)
    ax.set_title(f"origem {ends[tr[k]]} (réguas v1/05: patchtst 0,1432 · dlinear 0,1435)")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

# --- 05-mae: barras pooled (baratos + 2 modelos × 5 seeds + médias) ---
fig, ax = plt.subplots(figsize=(8, 5))
bar = pd.concat([pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T["MAE"],
                 tab_seed.set_index(["modelo", "seed"])["MAE"].set_axis([f"{m}_s{s}" for m, s in zip(tab_seed["modelo"], tab_seed["seed"])]),
                 pd.Series({f"{m}_media": tab_seed.loc[tab_seed["modelo"] == m, "MAE"].mean()
                            for m in MODELOS_N})]).sort_values()
bar.plot.barh(ax=ax)
ax.set_title("MAE na val pooled (5 fatias, v2) — baratos + PatchTST/DLinear por seed (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

# --- 06-val-dias: MAE por dia-âncora (baratos + médias ±dp) ---
fig, ax = plt.subplots(figsize=(12, 3.5))
pdf = por_dia
for col, ls in [("sazonal_naive_288", "--"), ("patchtst_media", "-"), ("dlinear_media", "-."), ("persistencia", ":")]:
    if col in pdf.columns:
        ax.plot(pd.to_datetime(pdf.index), pdf[col], ls, lw=1.1, label=col)
for col in ["patchtst_media", "dlinear_media"]:
    ax.fill_between(pd.to_datetime(pdf.index), pdf[col] - pdf[col.replace("media", "dp")],
                    pdf[col] + pdf[col.replace("media", "dp")], alpha=0.15)
ax.set_title("od — MAE por dia-âncora na val (5 fatias sazonais, v2, incl. dez)")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")

# --- 07-curvas-treino: por modelo, 5 seeds (finas) + média±dp (espessa + banda) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=False)
for ax, nome in zip(axes, MODELOS_N):
    for sd in SEEDS:
        ax.plot(hists[(nome, sd)]["val"], lw=0.8, alpha=0.5, label=f"val s{sd}")
    Lmax = max(len(hists[(nome, sd)]["val"]) for sd in SEEDS)
    arr_tr = np.full((len(SEEDS), Lmax), np.nan)
    arr_va = np.full((len(SEEDS), Lmax), np.nan)
    for j, sd in enumerate(SEEDS):
        arr_tr[j, :len(hists[(nome, sd)]["train"])] = hists[(nome, sd)]["train"]
        arr_va[j, :len(hists[(nome, sd)]["val"])] = hists[(nome, sd)]["val"]
    ep = np.arange(1, Lmax + 1)
    ax.plot(ep, np.nanmean(arr_tr, axis=0), "k-", lw=1.5, label="treino média")
    ax.plot(ep, np.nanmean(arr_va, axis=0), "r-", lw=1.5, label="val média")
    ax.fill_between(ep, np.nanmean(arr_va, axis=0) - np.nanstd(arr_va, axis=0, ddof=1),
                    np.nanmean(arr_va, axis=0) + np.nanstd(arr_va, axis=0, ddof=1),
                    color="r", alpha=0.2, label="val ±dp")
    ax.set_title(f"{nome} v2 — loss por época (5 seeds)")
    ax.set_xlabel("época"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-curvas-treino.png")
print("figs salvas")

figs salvas


## 11. Conclusões (preencher com números reais após a execução)

Réguas v2 acima (`metricas_val_por_seed.csv` = primária por seed; `metricas_val_media_dp.csv` = pooled média±dp por modelo; `metricas_por_fatia_media_dp.csv` mostra cada fatia×modelo, incl. dez). Réguas v1 de referência: **05 patchtst 0,1432 · dlinear 0,1435 · lstnet(03) 0,1380** (L=8640, 4 fatias sem purge) — a comparação v1×v2 embute mudança de protocolo (L, purge, 5ª fatia, strides), não só o modelo. Contexto LSTNet-v2 via CSV do 13 quando executado. Checkpoints por seed em `modelos/` para o benchmark/NNLS futuro.

### Protocolo v2 (resumo p/ o README do experimento)

- Janelas `L=2304 → H=288` (8 d → 1 d, 5 min), interp `time` limite 24, descarte com NaN; val 5 fatias por data de fim (19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]**); purge/embargo ±H (gap mín +289; trava por `assert`).
- Modelos = PatchTST + DLinearLite do 05, verbatim e univariados (cauda `LN=2016`, RevIN por janela; covariáveis só no LSTNet-v2/LGBM-v2 — limitação declarada no cabeçalho); strides 8/4→4/4 (idêntico ao 12/13).
- Treino = hiperparâmetros/early-stopping do 05 por (modelo, seed) (patch `BATCH=256/LR=1e-3/MAX=60/PAT=10`; dlinear `batch=512/30/5`, Adam/MSE, strides 4/4), seeds `[42, 7, 123, 2024, 999]`; reporte por seed + média±dp pooled, por fatia e por dia-âncora.
- OD só tem micro-outages (336 slots NaN pós-interp) → 5 fatias cheias, asserts de cobertura idênticos aos do 11/13.

### Procedência da execução (preencher no commit da execução)

- Host remoto: `temporal-remote` 192.168.1.6 · work dir: `/home/marcos/temporal-model` · data: `2026-09-17` · threads: `solo — 12c, threads unset, torch CPU 2.14.0+cpu`
- Pós-execução: escrever `resultados/15-v2-patchtst-od/README.md` (formato do 05 + seção “Protocolo v2”), indexar em `resultados/README.md` + `notebooks/README.md` + README §7 — com números reais. Não commitar `modelos/*.pt` (vão ao Release via `scripts/baixar_modelos.sh`).